# Fase 3.2: Modelado - Aprendizaje no supervisado

In [1]:
# Importar dependencias.
import constants
import gc
import numpy as np
import optuna
import os
import warnings
from joblib import dump
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.svm import OneClassSVM
from util import csv_util, date_util

# No mostrar mensajes de advertencia.
optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings('ignore')

## 1. Introducción

Un modelo de aprendizaje no supervisado procesa un conjunto de datos no etiquetados y, por tanto, su objetivo es analizar y clasificar dichos datos.

In [2]:
def load_dataset(city):
    df = csv_util.read(f'dataset/processed_data/{city}_dataset.csv', usecols=['date', 'city', 'section', 'roadway', 'direction', 'lane', 'heavy_traffic', 'max_pavement_temperature', 'min_pavement_temperature',
                                                                              'uv_index', 'precipitation', 'total_stress'],
                       dtype={'city':str, 'section':int, 'roadway':int, 'direction':int, 'lane':int, 'heavy_traffic':float, 'max_pavement_temperature':float, 'min_pavement_temperature':float,
                              'uv_index':float, 'precipitation':float, 'total_stress':float},
                       date_formats=[('date', '%Y-%m-%d')])

    # Añadir codificación cíclica para días de la semana y año.
    df = date_util.add_cyclic_coding(df)

    return df

def save_model(model, model_name, city):
    path = f'model/{model_name}/trial/{city}_{model_name}.pkl'

    if os.path.exists(path):
        os.remove(path)

    dump(model, path)

## 2. Isolation forest

In [3]:
def objective_isolation_forest(trial, X):
    # Hiperparámetros a optimizar.
    n_estimators = trial.suggest_int('n_estimators', 200, constants.MAX_ITERATIONS, step=100)
    max_samples = trial.suggest_float('max_samples', 0.5, 1.0)
    contamination = trial.suggest_float('contamination', 0.005, 0.05)
    max_features = trial.suggest_float('max_features', 0.6, 1.0)

    if_model = IsolationForest(n_estimators=n_estimators, max_samples=max_samples, contamination=contamination,
                               max_features=max_features, random_state=42, n_jobs=-1)

    if_model.fit(X)
    scores = if_model.decision_function(X)

    score_variance = np.var(scores)
    print(f'Trial: {trial.number + 1} | Puntuación: {score_variance:.4f}')

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del if_model
    del scores
    gc.collect()

    return score_variance

def fit_isolation_forest(study, X):
    best_params = study.best_params
    if_model = IsolationForest(n_estimators=best_params['n_estimators'], max_samples=best_params['max_samples'], contamination=best_params['contamination'],
                               max_features=best_params['max_features'], random_state=42, n_jobs=-1)
    if_model.fit(X)

    return if_model

def save_isolation_forest(if_model, df, X, city):
    df['anomaly_flag'] = if_model.predict(X)
    df['anomaly_score'] = if_model.decision_function(X)
    df = df[['date', 'city', 'section', 'roadway', 'direction', 'lane', 'heavy_traffic', 'max_pavement_temperature',
             'min_pavement_temperature', 'uv_index', 'precipitation', 'total_stress', 'anomaly_flag', 'anomaly_score']]
    csv_util.save(df, f'model/Isolation_Forest/result/{city}_Isolation_Forest_results.csv', mode='w')

In [5]:
def run_isolation_forest(city):
    print(f'{city.capitalize()} con isolation forest')
    print('1. Añadir características temporales')
    df = load_dataset(city)
    feature_cols = ['heavy_traffic', 'max_pavement_temperature', 'min_pavement_temperature', 'uv_index', 'precipitation']
    X = df[feature_cols].values

    print('2. Buscar mejores hiperparámetros con Optuna')
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective_isolation_forest(trial, X), n_trials=constants.NUM_TRIALS)

    print('3. Entrenar con los mejores hiperparámetros')
    if_model = fit_isolation_forest(study, X)
    save_model(if_model, 'Isolation_Forest', city)

    print('4. Guardar resultados')
    save_isolation_forest(if_model, df, X, city)

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del if_model
    gc.collect()
    print('FIN isolation forest\n')

for city in constants.CITIES:
    run_isolation_forest(city)

Auckland con isolation forest
1. Añadir características temporales
2. Buscar mejores hiperparámetros con Optuna
Trial: 1 | Puntuación: 0.0012
Trial: 2 | Puntuación: 0.0012
Trial: 3 | Puntuación: 0.0015
Trial: 4 | Puntuación: 0.0015
Trial: 5 | Puntuación: 0.0012
Trial: 6 | Puntuación: 0.0011
Trial: 7 | Puntuación: 0.0015
Trial: 8 | Puntuación: 0.0011
Trial: 9 | Puntuación: 0.0011
Trial: 10 | Puntuación: 0.0012
Trial: 11 | Puntuación: 0.0016
Trial: 12 | Puntuación: 0.0015
Trial: 13 | Puntuación: 0.0016
Trial: 14 | Puntuación: 0.0016
Trial: 15 | Puntuación: 0.0014
Trial: 16 | Puntuación: 0.0016
Trial: 17 | Puntuación: 0.0016
Trial: 18 | Puntuación: 0.0016
Trial: 19 | Puntuación: 0.0016
Trial: 20 | Puntuación: 0.0016
3. Entrenar con los mejores hiperparámetros
4. Guardar resultados
FIN isolation forest

Christchurch con isolation forest
1. Añadir características temporales
2. Buscar mejores hiperparámetros con Optuna
Trial: 1 | Puntuación: 0.0011
Trial: 2 | Puntuación: 0.0013
Trial: 3 | Pu

## 3. One-Class SVM

In [6]:
def apply_scale(df):
    # Estandarizar características con valores que siguen una distribución normal, donde el centro es la media.
    standard_scaler = StandardScaler()
    standard_scaler_cols = ['heavy_traffic', 'max_pavement_temperature', 'min_pavement_temperature']
    df[standard_scaler_cols] = standard_scaler.fit_transform(df[standard_scaler_cols])

    # Normalizar características con valores con límites bien definidos.
    min_max_scaler = MinMaxScaler()
    min_max_scaler_cols = ['uv_index']
    df[min_max_scaler_cols] = min_max_scaler.fit_transform(df[min_max_scaler_cols])

    # Escalar características con muchos valores iguales a 0.
    robust_scaler = RobustScaler()
    robust_scaler_cols = ['precipitation']
    df[robust_scaler_cols] = robust_scaler.fit_transform(df[robust_scaler_cols])

    return df.values

def objective_one_class_svm(trial, X_scaled):
    # Hiperparámetros a optimizar.
    nu_val = trial.suggest_float('nu', 0.005, 0.05)
    gamma_val = trial.suggest_categorical('gamma', ['scale', 'auto', 0.001, 0.01, 0.1, 1.0])

    ons_model = OneClassSVM(nu=nu_val, kernel='rbf', gamma=gamma_val)
    preds = ons_model.fit_predict(X_scaled)

    anomaly_rate = np.mean(preds == -1)
    target_rate = 0.02
    penalty = abs(anomaly_rate - target_rate)
    decision_scores = ons_model.decision_function(X_scaled)
    score_variance = np.var(decision_scores)
    loss = penalty - (0.001 * score_variance)
    print(f'Trial: {trial.number + 1} | Pérdida: {loss:.4f}')

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del ons_model
    gc.collect()

    return loss

def fit_one_class_svm(study, X_scaled):
    best_params = study.best_params
    ons_model = OneClassSVM(nu=best_params['nu'], kernel='rbf', gamma=best_params['gamma'])
    ons_model.fit(X_scaled)

    return ons_model

def save_one_class_svm(ons_model, df, X_scaled, city):
    df['oc_svm_pred'] = ons_model.predict(X_scaled)
    df['oc_svm_anomaly'] = np.where(df['oc_svm_pred'] == -1, 1, 0)
    df['oc_svm_score'] = ons_model.decision_function(X_scaled)
    df = df[['date', 'city', 'section', 'roadway', 'direction', 'lane', 'heavy_traffic', 'max_pavement_temperature',
             'min_pavement_temperature', 'uv_index', 'precipitation', 'total_stress', 'oc_svm_anomaly', 'oc_svm_score']]
    csv_util.save(df, f'model/One-Class_SVM/result/{city}_One-Class_SVM_results.csv', mode='w')

In [7]:
def run_one_class_svm(city):
    print(f'{city.capitalize()} con one-class SVM')
    print('1. Añadir características temporales')
    df = load_dataset(city)
    feature_cols = ['heavy_traffic', 'max_pavement_temperature', 'min_pavement_temperature', 'uv_index', 'precipitation']
    X_scaled = apply_scale(df[feature_cols].copy())

    print('2. Buscar mejores hiperparámetros con Optuna')
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective_one_class_svm(trial, X_scaled), n_trials=constants.NUM_TRIALS)

    print('3. Entrenar con los mejores hiperparámetros')
    ons_model = fit_one_class_svm(study, X_scaled)
    save_model(ons_model, 'One-Class_SVM', city)

    print('4. Guardar resultados')
    save_one_class_svm(ons_model, df, X_scaled, city)

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del ons_model
    gc.collect()
    print('FIN one-class SVM\n')

for city in constants.CITIES:
    run_one_class_svm(city)

Auckland con one-class SVM
1. Añadir características temporales
2. Buscar mejores hiperparámetros con Optuna
Trial: 1 | Pérdida: 0.0043
Trial: 2 | Pérdida: -0.0503
Trial: 3 | Pérdida: -2.5276
Trial: 4 | Pérdida: -0.0476
Trial: 5 | Pérdida: 0.0166
Trial: 6 | Pérdida: -0.3853
Trial: 7 | Pérdida: -7.4222
Trial: 8 | Pérdida: -4.7261
Trial: 9 | Pérdida: 0.0190
Trial: 10 | Pérdida: -0.0557
Trial: 11 | Pérdida: -1.7128
Trial: 12 | Pérdida: -5.9398
Trial: 13 | Pérdida: -8.3254
Trial: 14 | Pérdida: -8.5364
Trial: 15 | Pérdida: -0.3254
Trial: 16 | Pérdida: -0.5767
Trial: 17 | Pérdida: -9.1843
Trial: 18 | Pérdida: -4.5351
Trial: 19 | Pérdida: -11.4430
Trial: 20 | Pérdida: -17.4094
3. Entrenar con los mejores hiperparámetros
4. Guardar resultados
FIN one-class SVM

Christchurch con one-class SVM
1. Añadir características temporales
2. Buscar mejores hiperparámetros con Optuna
Trial: 1 | Pérdida: -0.4322
Trial: 2 | Pérdida: 0.0128
Trial: 3 | Pérdida: -3.4494
Trial: 4 | Pérdida: -0.9555
Trial: 5 | P